# Лабораторная работа 3

**Тема:** нахождение промежуточных значений функции, заданной в n точках, и построение графиков интерполяции.

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

Реализованы четыре способа:
1. кусочно-линейная интерполяция,
2. кусочно-параболическая интерполяция (по трём ближайшим узлам),
3. полином Лагранжа по итерационной схеме Эйткена (по заданию — для самостоятельного разбора,
   но реализован здесь же для полноты и для сверки с остальными методами),
4. кубический сплайн (`scipy.interpolate.CubicSpline`, встроенный модуль).


## Импорт библиотек


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline


## Ввод узлов интерполяции и искомой точки

По умолчанию используются узлы функции `sin(x)` на отрезке `[0, 5]` — этого достаточно, чтобы
проверить работу всех четырёх методов и сравнить их между собой.


In [ ]:
default_x_nodes = '0 1 2 3 4 5'
default_y_nodes = '0 0.84 0.91 0.14 -0.76 -0.96'

x_raw = input(f'Введите узлы x через пробел [{default_x_nodes}]: ').strip() or default_x_nodes
y_raw = input(f'Введите значения y через пробел [{default_y_nodes}]: ').strip() or default_y_nodes
x = float(input('Введите точку, где найти промежуточное значение [2.5]: ').strip() or '2.5')

x_nodes = np.array([float(v) for v in x_raw.split()], dtype=float)
y_nodes = np.array([float(v) for v in y_raw.split()], dtype=float)

if len(x_nodes) != len(y_nodes):
    raise ValueError('Количество x и y должно совпадать')
if len(x_nodes) < 3:
    raise ValueError('Для всех методов нужно минимум 3 точки')
if not np.all(np.diff(x_nodes) > 0):
    raise ValueError('Узлы x должны идти по возрастанию')
if not (x_nodes[0] <= x <= x_nodes[-1]):
    raise ValueError('Искомая точка должна лежать внутри интервала узлов')

print('Узлы интерполяции:')
for xi, yi in zip(x_nodes, y_nodes):
    print(f'x={xi:.1f}, y={yi:.2f}')
print(f'Искомая точка: x={x}')


## Методы интерполяции

- **Кусочно-линейная** — на отрезке `[x_i, x_i+1]`, куда попадает `x`, строится прямая `y = a*x + b`.
- **Кусочно-параболическая** — берутся 3 узла, ближайшие к `x` (не обязательно окружающие его с
  двух сторон поровну), через них проводится парабола, коэффициенты находятся решением системы 3×3.
- **Лагранж по схеме Эйткена** — промежуточные значения `L` последовательно уточняются по всем
  узлам; после `n` итераций `L[0]` равен значению полинома Лагранжа степени `n` в точке `x`.
- **Кубический сплайн** — гладкая кусочно-кубическая функция с естественными краевыми условиями
  (вторая производная равна нулю на концах).


In [ ]:
def piecewise_linear(x, x_nodes, y_nodes):
    for i in range(len(x_nodes) - 1):
        if x_nodes[i] <= x <= x_nodes[i + 1]:
            a = (y_nodes[i + 1] - y_nodes[i]) / (x_nodes[i + 1] - x_nodes[i])
            b = y_nodes[i] - a * x_nodes[i]
            return a * x + b
    raise ValueError("x находится вне диапазона заданных точек")


def piecewise_parabolic(x, x_nodes, y_nodes):
    indexes = np.argsort(np.abs(x_nodes - x))[:3]
    indexes = np.sort(indexes)
    xs = x_nodes[indexes]
    ys = y_nodes[indexes]
    A = np.array([[xs[0] ** 2, xs[0], 1], [xs[1] ** 2, xs[1], 1], [xs[2] ** 2, xs[2], 1]])
    a, b, c = np.linalg.solve(A, ys)
    return a * x ** 2 + b * x + c


def lagrange_aitken(x, x_nodes, y_nodes):
    n = len(x_nodes)
    L = y_nodes.copy().astype(float)
    for k in range(1, n):
        for i in range(n - k):
            j = i + k
            L[i] = ((x - x_nodes[i]) * L[i + 1] - (x - x_nodes[j]) * L[i]) / (x_nodes[j] - x_nodes[i])
    return L[0]


def cubic_spline(x, x_nodes, y_nodes):
    spline = CubicSpline(x_nodes, y_nodes, bc_type="natural")
    return spline(x)


## Промежуточное значение в точке x всеми четырьмя методами


In [ ]:
results = {
    "Кусочно-линейная": piecewise_linear(x, x_nodes, y_nodes),
    "Кусочно-параболическая": piecewise_parabolic(x, x_nodes, y_nodes),
    "Лагранж по Эйткену": lagrange_aitken(x, x_nodes, y_nodes),
    "Кубический сплайн": cubic_spline(x, x_nodes, y_nodes),
}
for name, value in results.items():
    print(f"{name:28s}: y = {value:.8f}")


## Теоретические оценки погрешности

По заданию для локальных методов оцениваются погрешности:

* кусочно-линейная: $R_1(x) = \dfrac{M_2}{8}h^2$, где $M_2 = \max|f''(x)|$, $h = x_i - x_{i-1}$;
* кусочно-параболическая: $R_2(x) = (x - x_0)(x - x_1)(x - x_2)\,\dfrac{M_3}{6}$, где $M_3 = \max|f'''(x)|$.

Функция задана только таблицей, поэтому $M_2$ и $M_3$ оцениваются по узлам через разделённые разности:
$f[x_{i-1}, x_i, x_{i+1}] = f''(\xi)/2!$, $\;f[x_i, \dots, x_{i+3}] = f'''(\xi)/3!$.

In [ ]:
def divided_differences(x_nodes, y_nodes, order):
    d = y_nodes.astype(float).copy()
    for k in range(1, order + 1):
        d = (d[1:] - d[:-1]) / (x_nodes[k:] - x_nodes[:-k])
    return d


M2 = 2 * np.max(np.abs(divided_differences(x_nodes, y_nodes, 2)))
M3 = 6 * np.max(np.abs(divided_differences(x_nodes, y_nodes, 3))) if len(x_nodes) >= 4 else float('nan')

i = min(np.searchsorted(x_nodes, x, side='right'), len(x_nodes) - 1)
h = x_nodes[i] - x_nodes[i - 1]
R1 = M2 / 8 * h ** 2

idx = np.sort(np.argsort(np.abs(x_nodes - x))[:3])
R2 = abs(np.prod(x - x_nodes[idx])) * M3 / 6

print(f'Оценка M2 = max|f\'\'| ≈ {M2:.4f},  M3 = max|f\'\'\'| ≈ {M3:.4f}')
print(f'Кусочно-линейная:       отрезок [{x_nodes[i - 1]:g}; {x_nodes[i]:g}], h = {h:g},  R1 = M2·h²/8 = {R1:.4f}')
print(f'Кусочно-параболическая: узлы {x_nodes[idx]},  R2 = |(x−x0)(x−x1)(x−x2)|·M3/6 = {R2:.4f}')

## Графики


In [ ]:
x_plot = np.linspace(x_nodes[0], x_nodes[-1], 500)
spline = CubicSpline(x_nodes, y_nodes, bc_type="natural")

plt.figure(figsize=(10, 6))
plt.scatter(x_nodes, y_nodes, label="Исходные точки", zorder=5)
plt.plot(x_plot, [piecewise_linear(t, x_nodes, y_nodes) for t in x_plot], label="Кусочно-линейная")
plt.plot(x_plot, [piecewise_parabolic(t, x_nodes, y_nodes) for t in x_plot], label="Кусочно-параболическая")
plt.plot(x_plot, [lagrange_aitken(t, x_nodes, y_nodes) for t in x_plot], label="Лагранж по Эйткену")
plt.plot(x_plot, spline(x_plot), label="Кубический сплайн")
plt.scatter([x], [results["Лагранж по Эйткену"]], marker="x", s=100, label=f"Искомая точка x={x}")
plt.title("Интерполяция функции по заданным точкам")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.legend()
plt.show()


## Вывод

Локальные методы (кусочно-линейный и кусочно-параболический) используют только ближайшие к `x`
узлы, поэтому дешевле в вычислении, но менее гладкие на стыках отрезков. Полином Лагранжа —
глобальный метод, использующий сразу все узлы: он точнее внутри области определения, но
неустойчив при большом числе узлов (эффект Рунге). Кубический сплайн даёт наиболее гладкое
приближение (непрерывны первая и вторая производные) и на практике — лучший компромисс между
точностью и устойчивостью.
